### Offline calibration flow for White Balance Correction (WBC) and Color Correction Matrix (CCM) coefficients.

In [ ]:
# *******************************************************************************
# Copyright (C) Altera Corporation
#
# This code and the related documents are Altera copyrighted materials and your
# use of them is governed by the express license under which they were provided to
# you ("License"). This code and the related documents are provided as is, with no
# express or implied warranties other than those that are expressly stated in the 
# License.
# *******************************************************************************/

In [1]:
from IPython.display import display, HTML
display(HTML("<style>:root { --jp-notebook-max-width: 75% !important; }</style>"))

import os
import numpy as np
import cv2
from matplotlib import pyplot as plt
%matplotlib widget
from PIL import Image
import colour
import isp.isp as isp

In [2]:
# sensor = 'imx519'
# sensor = 'imx477'
sensor = 'imx678'

bits_file = 16
if sensor == 'imx519':
    bits_capture = 10
else:
    bits_capture = 12

if sensor == 'imx477':
    black_level_pedestals = [[256, 256, 256, 256]]
elif sensor == 'imx678':
    black_level_pedestals = [[200, 200, 200, 200]]
elif sensor == 'imx519':
    black_level_pedestals = [[64, 64, 64, 64]]

roteate180 = True

In [ ]:
# Independent parameters and constants
cfa_pattern = 'RGGB'

# corner coordinates of the color chart (in the order: vertical_start, vertical_end, horizontal_start, horizontal_end)
# Make sure the CFA pattern of the corpped area is RGGB
if sensor == 'imx477':
    cbox = (271, 690, 1467, 2098)
elif sensor == 'imx678':
    cbox = (794, 1195, 1616, 2223) # Start with odd end with even to accomodate RGGB after rotation
elif sensor == 'imx519':
    cbox = (136, 723, 480, 1357) # Start with odd end with even to accomodate RGGB after rotation
print(cbox)
cpatch_active = 0.33 # average center 33% x 33% of the pixels of each color patch

ref_lab = np.array([ # reference 24 colors of the color chart in L*a*b* color space
    [[37.54,  14.37,  14.92]], [[64.66,  19.27,  17.50]], [[49.32,  -3.82, -22.54]], [[43.46, -12.74,  22.72]], [[54.94,   9.61, -24.79]], [[70.48, -32.26,  -0.37]],
    [[62.73,  35.83,  56.50]], [[39.43,  10.75, -45.17]], [[50.57,  48.64,  16.67]], [[30.10,  22.54, -20.87]], [[71.77, -24.13,  58.19]], [[71.51,  18.24,  67.37]],
    [[28.37,  15.42, -49.80]], [[54.38, -39.72,  32.27]], [[42.43,  51.05,  28.62]], [[81.80,   2.67,  80.41]], [[50.63,  51.28, -14.12]], [[49.57, -29.71, -28.32]],
    [[95.19,  -1.03,   2.93]], [[81.29,  -0.57,   0.44]], [[66.89,  -0.75,  -0.06]], [[50.76,  -0.13,   0.14]], [[35.63,  -0.46,  -0.48]], [[20.64,   0.07,  -0.46]],
])

# Derived parameters and constants
ch = cbox[1] - cbox[0] # height of the color chart in pixels
cw = cbox[3] - cbox[2] # width of the color chart in pixels
chstep = ch / 4 # horizontal distance between color patches in pixels
cwstep = cw / 6 # vertical distance between color patches in pixels

ref_srgb = colour.XYZ_to_sRGB(colour.Lab_to_XYZ(ref_lab))
ref_lin  = colour.models.eotf(ref_srgb, 'sRGB')

folder = './' + sensor + '_calibration/wb_ccm/'
for root, dirs, files in os.walk(folder):
    files.sort()
    for file in files:
        if file[-4:] != '.tif' and file[-5:] != '.tiff':
            print('Skipping non-TIFF file ' + file)
            continue
        # Read color temperature from the file name
        ctemp = file[:5]
        print("\n", ctemp)
        # Read the image with OpenCV (PIL does not support 48 bit tiff files)
        # Also rotate the image 180 degree and convert BGR to RGB
        img_lin_cfa = cv2.imread(folder+file, cv2.IMREAD_UNCHANGED)
        assert len(img_lin_cfa.shape)==2, "Color calibration expects 2D arrays (CFA images)."
        img_lin_cfa = img_lin_cfa / 2.0**(bits_file - bits_capture)
        img_lin_cfa = img_lin_cfa[cbox[0]:cbox[1], cbox[2]:cbox[3]]
        if roteate180:
            img_lin_cfa = img_lin_cfa[::-1, ::-1]

        # Correct black level
        img_lin_cfa[ ::2, ::2] -= black_level_pedestals[0][0]
        img_lin_cfa[ ::2,1::2] -= black_level_pedestals[0][1]
        img_lin_cfa[1::2, ::2] -= black_level_pedestals[0][2]
        img_lin_cfa[1::2,1::2] -= black_level_pedestals[0][3]
        img_lin_cfa = np.abs(img_lin_cfa)
        
        # Demosaic the CFA image
        img_lin_cfa_pad = np.pad(img_lin_cfa, 2)
        img_lin_cfa_pad[  :2,  : ] = img_lin_cfa_pad[ 2: 4,  :  ]
        img_lin_cfa_pad[-2: ,  : ] = img_lin_cfa_pad[-4:-2,  :  ]
        img_lin_cfa_pad[  : ,  :2] = img_lin_cfa_pad[  :  , 2: 4]
        img_lin_cfa_pad[  : ,-2: ] = img_lin_cfa_pad[  :  ,-4:-2]
        img_lin = np.abs(isp.demosaic_malvar(img_lin_cfa_pad, cfa_pattern)[2:-2,2:-2])
        
        # Scale to [0.0, 1.0] range
        img_lin /= 2**bits_capture
        
        # print(img_lin.min())
        # print(img_lin.max())
        # plt.figure(figsize=(9, 6)); plt.imshow(img_lin_cfa)
        # plt.figure(figsize=(9, 6)); plt.imshow(img_lin)
        # sdf

        # Read 24 color patches from the color chart cropped form the image.
        # Average colors over a smaller center portion of each patch to avoid edges.
        src_lin = np.zeros_like(ref_lab)
        wbc = np.zeros(3)
        for j in range(4):
            for i in range(6):
                x = int(cwstep * (j + cpatch_active))
                y = int(chstep * (i + cpatch_active))
                src_lin[j*6+i,0] = np.mean(img_lin[x:x+int(chstep * cpatch_active), y:y+int(cwstep * cpatch_active)], axis=(0,1))
                # Calculate White Balance Coefficients (WBC)
                if j == 3:
                    gray = src_lin[j*6+i,0]
                    wbc += np.array([
                        gray[1] / gray[0], 1.0, gray[1] / gray[2]
                    ])
        wbc /= 6
        wbc_t = wbc[:, np.newaxis, np.newaxis]
        wbc_t = np.transpose(wbc_t, (1, 2, 0))
    
        # Calculate Color Correction Matrix (CCM).
        # Slice the last color patch out since the reference color chart was damaged.
        # For linear RGB domain in HDR use COLOR_SPACE_REC_2020_RGBL, otherwise COLOR_SPACE_sRGBL.
        model = cv2.ccm.ColorCorrectionModel(src_lin[:-1], ref_lin[:-1], cv2.ccm.COLOR_SPACE_sRGBL);
        if False: # Offset is not used in the ISP flow, so CCM is 3x3
            model.setCCM_TYPE(cv2.ccm.CCM_4x3)
        model.setLinear(cv2.ccm.LINEARIZATION_IDENTITY) # Force the computation domain to linear
        model.run()
        ccm_combined = model.getCCM().T
        print("Combined CCM and WB matrix for " + ctemp + ":")
        print(ccm_combined.round(4))
        print("Loss reported by OpenCV:", model.getLoss())

        # Disentangle exposure normlization factor, WB and CCM
        ccm = 1/wbc * ccm_combined
        # Calculate the amount of underexposure as a normalization factor
        exp_norm = ccm.sum(axis=1).mean()
        # Normalize CCM
        ccm /= exp_norm
        ccm_unity_error = 1.0 - ccm.sum(axis=1)
        ccm += np.diag(ccm_unity_error)
        print("White balance coefficients:", wbc, wbc_t)
        print("CCM:\n", ccm.round(4))
        print("CCM unity:", ccm.sum(axis=1))
        print("Exposure normalization factor:", exp_norm)

        # ccm = exp_norm * wbc * ccm
        print("Error of uncombining: ", abs(exp_norm * wbc * ccm - ccm_combined).max().round(4))

        # Apply CCM. Clip the result since out of range colors are possible in calculation.
        img_lin_exp = exp_norm * img_lin
        img_lin_wbc = wbc_t * img_lin_exp
        img_lin_ccm = np.einsum('ij,...j', ccm, img_lin_wbc, optimize=True)
        img_lin_ccm[img_lin_ccm<0.0] = 0.0
        img_lin_ccm[img_lin_ccm>1.0] = 1.0
        if False: # use sRGB for now for SDR
            img_oetf = colour.oetf(img_lin, 'ITU-R BT.709') # 'ITU-R BT.709' for SDR; 'ITU-R BT.2100 HLG' or 'ITU-R BT.2100 PQ' for HDR
            img_oetf_ccm = colour.oetf(img_lin_ccm, 'ITU-R BT.709')
        else:
            img_oetf = colour.models.eotf_inverse_sRGB(img_lin) # Inverse EOTF of sRGB to match the CCM calibration domain
            img_oetf_wbc = colour.models.eotf_inverse_sRGB(img_lin_wbc)
            img_oetf_ccm = colour.models.eotf_inverse_sRGB(img_lin_ccm)

        # Calculate delta-E error in the L*a*b* domain
        img_lab_ccm = colour.XYZ_to_Lab(colour.sRGB_to_XYZ(img_oetf_ccm))
        dst_lab = np.zeros_like(ref_lab)
        for j in range(4):
            for i in range(6):
                x = int(cwstep * (j + 0.33))
                y = int(chstep * (i + 0.33))
                dst_lab[j*6+i,0] = np.mean(img_lab_ccm[x:x+int(chstep/3), y:y+int(cwstep/3)], axis=(0,1))
        # delta_E = colour.delta_E(ref_lab[::-1], dst_lab[::-1]) # Workaround: discarding the last gray patch that is damaged
        delta_E = colour.delta_E(ref_lab, dst_lab)
        print("Max delta-E:", np.max(delta_E))
        print("Average delta-E:", np.mean(delta_E))

        # Overlay reference colors in the center of each color patch
        img_oetf_ccm_overlayed = img_oetf_ccm.copy()
        for j in range(4):
            for i in range(6):
                x = int(cwstep * (j + 0.33))
                y = int(chstep * (i + 0.33))
                img_oetf_ccm_overlayed[x:x+int(chstep/3), y:y+int(cwstep/3)] = ref_srgb[j*6+i,0]
        img_oetf_ccm_overlayed[img_oetf_ccm_overlayed<0.0] = 0.0
        img_oetf_ccm_overlayed[img_oetf_ccm_overlayed>1.0] = 1.0

        # Visualize
        if False:
            plt.figure(figsize=(9, 6)); plt.imshow(img_oetf)
        if True:
            plt.figure(figsize=(9, 6)); plt.imshow(img_oetf_wbc)
        if False:
            plt.figure(figsize=(9, 6)); plt.imshow(img_oetf_ccm)
        if True:
            plt.figure(figsize=(9, 6)); plt.imshow(img_oetf_ccm_overlayed)
        
        print(3*"\t" + "\"rgb_scalars\":")
        print(3*"\t" + "[")
        for j in range(3):
            print(4*"\t", wbc[j], ("," if j < 2 else ""), sep="")
        print(3*"\t" + "],")
        print(3*"\t" + "\"ccm_coeffs\":")
        print(3*"\t" + "[")
        for i in range(3):
            print(4*"\t" + "[")
            for j in range(3):
                print(5*"\t", ccm.T[i,j], ",", sep="")
            print(5*"\t", 0.0, sep="")
            print(4*"\t" + "]" + ("," if i < 2 else ""))
        print(3*"\t" + "],")

        # break


(794, 1195, 1616, 2223)

 2700K
Combined CCM and WB matrix for 2700K:
[[  2.5876  -0.2289  -2.6225]
 [ -1.3829   2.8603  -1.4215]
 [ -0.1745  -2.4096  11.4062]]
Loss reported by OpenCV: 2.9494430240340015
White balance coefficients: [ 1.14811443  1.          3.11420713] [[[ 1.14811443  1.          3.11420713]]]
CCM:
 [[ 1.9225 -0.1972 -0.7253]
 [-1.0374  2.4305 -0.3931]
 [-0.1309 -2.0753  3.2062]]
CCM unity: [ 1.  1.  1.]
Exposure normalization factor: 1.16105982017
Error of uncombining:  0.1868
Max delta-E: 8.46680362131
Average delta-E: 3.15896207258
			"rgb_scalars":
			[
				1.14811443179,
				1.0,
				3.11420713426
			],
			"ccm_coeffs":
			[
				[
					1.92245450778,
					-1.03739186631,
					-0.130915367933,
					0.0
				],
				[
					-0.197165134116,
					2.43053466194,
					-2.07530799319,
					0.0
				],
				[
					-0.725289373663,
					-0.393142795629,
					3.20622336112,
					0.0
				]
			],

 3200K
Combined CCM and WB matrix for 3200K:
[[ 3.1218 -0.5188 -1.6117]
 [-1.3127 